# G1 Humanoid Robot LQR Balancing Control

This notebook implements Time-Varying Linear Quadratic Regulator (TVLQR) control for balancing the G1 humanoid robot using:
- GTDynamics for trajectory optimization and LQR policy computation
- MuJoCo for physics simulation
- Matplotlib (pyplot) for visualization

## Workflow:
1. Load G1 robot model
2. Build factor graph for balancing optimization
3. Optimize trajectory (standing still)
4. Linearize around trajectory to get Gaussian factor graph
5. Eliminate to get Bayes net (LQR policy)
6. Simulate with MuJoCo applying TVLQR feedback control
7. Visualize results

## 1. Imports and Setup

In [1]:
import numpy as np
import gtsam
import gtdynamics as gtd
import mujoco
import mujoco.viewer
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mediapy as media

print("Imports successful!")
print(f"GTDynamics version: {gtd.__version__ if hasattr(gtd, '__version__') else 'Unknown'}")
print(f"GTSAM version: {gtsam.__version__ if hasattr(gtsam, '__version__') else 'Unknown'}")
print(f"MuJoCo version: {mujoco.__version__ if hasattr(mujoco, '__version__') else 'Unknown'}")

Imports successful!
GTDynamics version: Unknown
GTSAM version: Unknown
MuJoCo version: 3.3.2


## 2. MuJoCo Viewer Playground
Interactive viewer to explore the G1 robot model before running optimization.

In [2]:
# MuJoCo Viewer Playground - Only run for interactive exploration
MJCF_PATH = '../../models/urdfs/g1_description/g1_23dof.xml'

model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

#mujoco.viewer.launch(model, data)

## 3. Load G1 Robot Model

In [3]:
# Load G1 robot from URDF
URDF_PATH = '../../models/urdfs/g1_description/g1_23dof.urdf'
robot = gtd.CreateRobotFromFile(URDF_PATH)

print(f"Number of links: {robot.numLinks()}")
print(f"Number of joints: {robot.numJoints()}")

# Print link names
print("\nLink names:")
for i, link in enumerate(robot.links()):
    print(f"{link.name()}: {link.id()}")
    print(link.bMlink())


# Print joint names
print("\nJoint names:")
for i, joint in enumerate(robot.joints()):
    print(f"  {i}: {joint.name()}")

Number of links: 24
Number of joints: 23

Link names:
left_ankle_pitch_link: 5
R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: -2.3261e-06    0.118506   -0.739306

left_ankle_roll_link: 6
R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: -2.3261e-06    0.118506   -0.756864

left_elbow_link: 17
R: [
	1, 0.000191593, 5.49375e-05;
	-0.00019159, 1, -6.00105e-05;
	-5.4949e-05, 6e-05, 1
]
t: 0.0157745  0.146808  0.105243

left_hip_pitch_link: 1
R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t:        0 0.064452  -0.1027

left_hip_roll_link: 2
R: [
	0.984744, -0, -0.17401;
	0, 1, -0;
	0.17401, 0, 0.984744
]
t:         0  0.116452 -0.133165

left_hip_yaw_link: 3
R: [
	0.984744, -0, -0.17401;
	0, 1, -0;
	0.17401, 0, 0.984744
]
t: 0.0462177  0.116452 -0.251041

left_knee_link: 4
R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: -2.3261e-06    0.118601   -0.439296

left_shoulder_pitch_link: 14
R: [
	1, 0.000199314, -4.17366e-10;
	-0.00019159, 0.961246, -0.275692;
	-5.4949e-05, 0.275692, 0.961246
]
t: -7.2e-06  0.10022  0.29178

left_shou

## 4. Trajectory Optimization Parameters

In [4]:
# Time parameters
balance_time = 3.0  # seconds - duration of balancing
balance_steps = 300  # number of time steps
balance_dt = balance_time / balance_steps  # time step duration

print(f"Optimization horizon: {balance_time} seconds")
print(f"Number of time steps: {balance_steps}")
print(f"Time step size (dt): {balance_dt:.4f} seconds")

# Get number of joints
pelvis_link_id = 0
num_links = robot.numLinks()
num_joints = robot.numJoints()

# Initial configuration - G1 standing pose (neutral position)
# All joints start at zero (neutral standing pose)
initial_joint_angles = np.zeros(num_joints)

print(f"\nInitial joint configuration set for {num_joints} joints (all zero - neutral pose)")

Optimization horizon: 3.0 seconds
Number of time steps: 300
Time step size (dt): 0.0100 seconds

Initial joint configuration set for 23 joints (all zero - neutral pose)


## 4.5 Define Contact Points

We'll define contact points at both feet to ensure proper ground contact during balancing.

In [5]:
# Define contact points at the feet
# We need to identify which links are the feet and define contact points

# Print all link names to identify foot links
print("Available links:")
for i, link in enumerate(robot.links()):
    print(f"  {i}: {link.name()}")

# For G1 humanoid, the foot links are typically named with "ankle" or "foot"
# Let's find them
foot_link_names = []
for link in robot.links():
    link_name = link.name()
    if 'ankle_roll' in link_name.lower():
        foot_link_names.append(link_name)
        print(f"\nFound foot link: {link_name} (id: {link.id()})")

contact_in_link_frame = np.array([0, 0.0, -0.03])

print(f"\nUsing single centroid contact point per foot at: {contact_in_link_frame}")

# Create list of contact points
contact_points = []
for link in robot.links():
    if link.name() in foot_link_names:
        contact_points.append(gtd.PointOnLink(link, contact_in_link_frame))
        print(f"Added single centroid contact point for {link.name()}")

print(f"\nTotal contact points: {len(contact_points)}") # This should now print 2

# Ground plane parameters
ground_plane_height = 0.0  # Ground at z=0
friction_coefficient = 1.0  # Friction coefficient for contact

Available links:
  0: left_ankle_pitch_link
  1: left_ankle_roll_link
  2: left_elbow_link
  3: left_hip_pitch_link
  4: left_hip_roll_link
  5: left_hip_yaw_link
  6: left_knee_link
  7: left_shoulder_pitch_link
  8: left_shoulder_roll_link
  9: left_shoulder_yaw_link
  10: left_wrist_roll_rubber_hand
  11: pelvis
  12: right_ankle_pitch_link
  13: right_ankle_roll_link
  14: right_elbow_link
  15: right_hip_pitch_link
  16: right_hip_roll_link
  17: right_hip_yaw_link
  18: right_knee_link
  19: right_shoulder_pitch_link
  20: right_shoulder_roll_link
  21: right_shoulder_yaw_link
  22: right_wrist_roll_rubber_hand
  23: torso_link

Found foot link: left_ankle_roll_link (id: 6)

Found foot link: right_ankle_roll_link (id: 12)

Using single centroid contact point per foot at: [ 0.    0.   -0.03]
Added single centroid contact point for left_ankle_roll_link
Added single centroid contact point for right_ankle_roll_link

Total contact points: 2


## 5. Build Factor Graph for Balancing

In [6]:
# Cost model parameters (noise models)
sigma_dynamics = 1e-3      # Hard constraint for dynamics/collocation
sigma_objectives = 1e-1    # Medium constraint for joint angle/velocity objectives
sigma_torque = 1e0        # Miduym constraint for torque minimization (needed for LQR linearization)
sigma_base = 1.0           # Very loose constraint for base pose/twist (soft constraint)

# Create noise models
dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics)
dynamics_model_3 = gtsam.noiseModel.Isotropic.Sigma(3, sigma_dynamics)
dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics)
objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives)
torque_model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_torque)
base_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_base)  # Soft base constraints

# Create optimization settings and graph builder
gravity_vec = np.array([0, 0, -9.81])
opt_params = gtd.OptimizerSetting(sigma_dynamics)
graph_builder = gtd.DynamicsGraph(opt_params, gravity_vec, None)

# Build the trajectory factor graph WITH contact points
# This automatically adds dynamics, kinematics, collocation constraints, AND contact constraints
graph = graph_builder.trajectoryFG(
    robot, 
    balance_steps, 
    balance_dt,
    gtd.CollocationScheme.Trapezoidal,  # Collocation scheme
    contact_points,  # Contact points at feet
    friction_coefficient  # Friction coefficient
)

print(f"Factor graph built with {graph.size()} factors (including contact constraints)")

Factor graph built with 58649 factors (including contact constraints)


## 5.1 Forward Kinematics

In [7]:
# Test forward kinematics with all joints at 0
# Use left_ankle_roll_link as the base (priorLink) and place it at the specified position
fk_values = gtsam.Values()

# Add all joint angles at 0
for j in range(robot.numJoints()):
    fk_values.insert(gtd.JointAngleKey(j, 0), 0.0)

# Find ankle links
left_ankle_link = None
right_ankle_link = None
pelvis_link = None
for link in robot.links():
    if link.name() == "left_ankle_roll_link":
        left_ankle_link = link
    elif link.name() == "right_ankle_roll_link":
        right_ankle_link = link
    elif link.name() == "pelvis":
        pelvis_link = link

if left_ankle_link is None:
    print("ERROR: Could not find left_ankle_roll_link")
else:
    # Place left ankle at the specified position from URDF
    # Position: [-2.3261e-06, 0.118506, -0.756864]
    # Rotation: Identity (no rotation)
    left_ankle_pose = gtsam.Pose3(
        gtsam.Rot3(),  # Identity rotation
        np.array([0.0, 0.0, 0.0])
    )
    fk_values.insert(gtd.PoseKey(left_ankle_link.id(), 0), left_ankle_pose)
    
    # Compute forward kinematics using left ankle as the prior link
    fk_result = robot.forwardKinematics(fk_values, 0, left_ankle_link.name())
    
    print("Forward Kinematics Results (all joints at 0, left_ankle_roll_link as base):")
    print("=" * 80)
    
    # Print left ankle position (should match what we set)
    print(f"\n{left_ankle_link.name()} (id: {left_ankle_link.id()}) [BASE/PRIOR LINK]:")
    left_ankle_result = gtd.Pose(fk_result, left_ankle_link.id(), 0)
    left_ankle_pos = left_ankle_result.translation()
    print(f"  Position: {left_ankle_pos}")
    print(f"  z-coordinate: {left_ankle_pos[2]:.6f} m")
    
    # Print right ankle position (should be at specified position)
    if right_ankle_link:
        print(f"\n{right_ankle_link.name()} (id: {right_ankle_link.id()}):")
        right_ankle_result = gtd.Pose(fk_result, right_ankle_link.id(), 0)
        right_ankle_pos = right_ankle_result.translation()
        print(f"  Position: {right_ankle_pos}")
        print(f"  z-coordinate: {right_ankle_pos[2]:.6f} m")
    
    # Print pelvis position (should be derived from FK)
    print(f"\npelvis (id: {pelvis_link.id()}):")
    pelvis_pose = gtd.Pose(fk_result, pelvis_link.id(), 0)
    pelvis_pos = pelvis_pose.translation()
    print(f"  Position: {pelvis_pos}")
    print(f"  z-coordinate: {pelvis_pos[2]:.6f} m")
    
    # Print all link positions
    print("\n" + "=" * 80)
    print("All Link Positions:")
    print("=" * 80)
    for link in robot.links():
        link_pose = gtd.Pose(fk_result, link.id(), 0)
        link_pos = link_pose.translation()
        print(f"{link.id():3d}: {link.name():30s} - Position: [{link_pos[0]:10.6f}, {link_pos[1]:10.6f}, {link_pos[2]:10.6f}]")

Forward Kinematics Results (all joints at 0, left_ankle_roll_link as base):

left_ankle_roll_link (id: 6) [BASE/PRIOR LINK]:
  Position: [0. 0. 0.]
  z-coordinate: 0.000000 m

right_ankle_roll_link (id: 12):
  Position: [ 0.         -0.23701291  0.        ]
  z-coordinate: 0.000000 m

pelvis (id: 0):
  Position: [-0.02650267 -0.11850645  0.69725869]
  z-coordinate: 0.697259 m

All Link Positions:
  5: left_ankle_pitch_link          - Position: [ -0.033774,   0.000000,   0.045120]
  6: left_ankle_roll_link           - Position: [  0.000000,   0.000000,   0.000000]
 17: left_elbow_link                - Position: [  0.054228,   0.032744,   0.868466]
  1: left_hip_pitch_link            - Position: [ -0.023762,  -0.006263,   0.644529]
  2: left_hip_roll_link             - Position: [  0.018156,  -0.003099,   0.558719]
  3: left_hip_yaw_link              - Position: [ -0.010876,  -0.013035,   0.363726]
  4: left_knee_link                 - Position: [ -0.021048,   0.004058,   0.213253]
 14: 

In [8]:
# Test forward kinematics with all joints at 0
# Use left_ankle_roll_link as the base (priorLink) and place it at the specified position
fk_values = gtsam.Values()

# Add all joint angles at 0
for j in range(robot.numJoints()):
    fk_values.insert(gtd.JointAngleKey(j, 0), 0.0)

# Find ankle links
left_ankle_link = None
right_ankle_link = None
pelvis_link = None
for link in robot.links():
    if link.name() == "left_ankle_roll_link":
        left_ankle_link = link
    elif link.name() == "right_ankle_roll_link":
        right_ankle_link = link
    elif link.name() == "pelvis":
        pelvis_link = link

if left_ankle_link is None:
    print("ERROR: Could not find left_ankle_roll_link")
else:
    # Place left ankle at the specified position from URDF
    # Position: [-2.3261e-06, 0.118506, -0.756864]
    # Rotation: Identity (no rotation)
    pelvis_link_pose = gtsam.Pose3(
        gtsam.Rot3(),  # Identity rotation
        np.array([0.0, 0.0, 0.793])
    )
    fk_values.insert(gtd.PoseKey(pelvis_link.id(), 0), pelvis_link_pose)
    
    # Compute forward kinematics using left ankle as the prior link
    fk_result = robot.forwardKinematics(fk_values, 0, "pelvis")
    
    print("Forward Kinematics Results (all joints at 0, pelvis as base):")
    print("=" * 80)
    
    # Print left ankle position (should match what we set)
    print(f"\n{"pelvis"} (id: {pelvis_link.id()}) [BASE/PRIOR LINK]:")
    pelvis_result = gtd.Pose(fk_result, pelvis_link.id(), 0)
    pelvis_pose = pelvis_result.translation()
    print(f"  Position: {pelvis_pose}")
    print(f"  z-coordinate: {pelvis_pose[2]:.6f} m")
    
    # Print right ankle position (should be at specified position)
    if right_ankle_link:
        print(f"\n{right_ankle_link.name()} (id: {right_ankle_link.id()}):")
        right_ankle_result = gtd.Pose(fk_result, right_ankle_link.id(), 0)
        right_ankle_pos = right_ankle_result.translation()
        print(f"  Position: {right_ankle_pos}")
        print(f"  z-coordinate: {right_ankle_pos[2]:.6f} m")
    
    # Print pelvis position (should be derived from FK)
    print(f"\left ankle (id: {left_ankle_link.id()}):")
    left_ankle_pose = gtd.Pose(fk_result, left_ankle_link.id(), 0)
    left_ankle_pos = left_ankle_pose.translation()
    print(f"  Position: {left_ankle_pos}")
    print(f"  z-coordinate: {left_ankle_pos[2]:.6f} m")
    
    # Print all link positions
    print("\n" + "=" * 80)
    print("All Link Positions:")
    print("=" * 80)
    for link in robot.links():
        link_pose = gtd.Pose(fk_result, link.id(), 0)
        link_pos = link_pose.translation()
        print(f"{link.id():3d}: {link.name():30s} - Position: [{link_pos[0]:10.6f}, {link_pos[1]:10.6f}, {link_pos[2]:10.6f}]")

Forward Kinematics Results (all joints at 0, pelvis as base):

pelvis (id: 0) [BASE/PRIOR LINK]:
  Position: [0.    0.    0.793]
  z-coordinate: 0.793000 m

right_ankle_roll_link (id: 12):
  Position: [ 0.02650267 -0.11850645  0.09574131]
  z-coordinate: 0.095741 m
\left ankle (id: 6):
  Position: [0.02650267 0.11850645 0.09574131]
  z-coordinate: 0.095741 m

All Link Positions:
  5: left_ankle_pitch_link          - Position: [ -0.007271,   0.118506,   0.140861]
  6: left_ankle_roll_link           - Position: [  0.026503,   0.118506,   0.095741]
 17: left_elbow_link                - Position: [  0.080731,   0.151251,   0.964208]
  1: left_hip_pitch_link            - Position: [  0.002741,   0.112243,   0.740270]
  2: left_hip_roll_link             - Position: [  0.044659,   0.115407,   0.654460]
  3: left_hip_yaw_link              - Position: [  0.015626,   0.105471,   0.459467]
  4: left_knee_link                 - Position: [  0.005455,   0.122565,   0.308994]
 14: left_shoulder_pitc

<>:55: SyntaxWarning: invalid escape sequence '\l'
<>:55: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_94231/1674625497.py:55: SyntaxWarning: invalid escape sequence '\l'
  print(f"\left ankle (id: {left_ankle_link.id()}):")


## 6. Add Boundary Conditions and Objectives

In [9]:
# Add boundary conditions and objectives

# Define desired left ankle pose (COM position when joints at 0)
desired_ankle_pose = gtsam.Pose3(gtsam.Rot3(), np.array([0.0, 0.1, 0.03]))
zero_twist = np.zeros(6)

# Soft objective models
soft_objectives_angle = gtsam.noiseModel.Isotropic.Sigma(1, 1.0)  # Soft for intermediate angles
soft_objectives_pose = gtsam.noiseModel.Isotropic.Sigma(6, 1.0)   # Soft for final ankle pose

# === INITIAL STATE (t=0) ===
# Hard constraints on initial conditions
for j in range(num_joints):
    graph.addPriorDouble(gtd.JointAngleKey(j, 0), 0.0, dynamics_model_1)  # All joints at 0
    graph.addPriorDouble(gtd.JointVelKey(j, 0), 0.0, dynamics_model_1)    # Zero velocity

# Fix left ankle position at initial state (hard constraint)
graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(left_ankle_link.id(), 0), desired_ankle_pose, dynamics_model_6))
graph.add(gtsam.PriorFactorVector(gtd.TwistKey(left_ankle_link.id(), 0), zero_twist, dynamics_model_6))

# === INTERMEDIATE STATES (t=1 to t=balance_steps-1) ===
# Soft objectives to encourage joints to stay near 0
for t in range(1, balance_steps):
    for j in range(num_joints):
        graph.addPriorDouble(gtd.JointAngleKey(j, t), 0.0, soft_objectives_angle)

# === FINAL STATE (t=balance_steps) ===
# Constrain joints back to 0 (for periodic trajectory)
for j in range(num_joints):
    graph.addPriorDouble(gtd.JointAngleKey(j, balance_steps), 0.0, soft_objectives_angle)
    graph.addPriorDouble(gtd.JointVelKey(j, balance_steps), 0.0, dynamics_model_1)

# Fix left ankle position at final state (soft - let dynamics adjust if needed)
graph.add(gtsam.PriorFactorPose3(gtd.PoseKey(left_ankle_link.id(), balance_steps), desired_ankle_pose, soft_objectives_pose))
graph.add(gtsam.PriorFactorVector(gtd.TwistKey(left_ankle_link.id(), balance_steps), zero_twist, soft_objectives_pose))

# === TORQUE MINIMIZATION ===
# Add minimum torque objectives (encourage small control effort)
"""
for t in range(balance_steps + 1):
    for j in range(num_joints):
        graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(j, t), torque_model))
"""

print(f"Added boundary conditions:")
print(f"  - Initial: all joints=0, left ankle at {desired_ankle_pose.translation()}")
print(f"  - Intermediate: soft goals for joints=0")
print(f"  - Final: all joints=0, left ankle at {desired_ankle_pose.translation()}")
print(f"Total factors in graph: {graph.size()}")


Added boundary conditions:
  - Initial: all joints=0, left ankle at [0.   0.1  0.03]
  - Intermediate: soft goals for joints=0
  - Final: all joints=0, left ankle at [0.   0.1  0.03]
Total factors in graph: 65622


## 7. Initialize Values and Optimize

In [10]:
# Initialize values using GTDynamics Initializer with contact points
initializer = gtd.Initializer()
init_values = initializer.ZeroValuesTrajectory(robot, balance_steps, 0, 0.0, contact_points)

# CRITICAL: Use FK to initialize kinematically consistent poses
# Strategy: Set left ankle at desired position, run FK to get all other link poses
print("Initializing kinematically consistent poses using FK...")
print(f"Using left_ankle_roll_link as prior at position: {desired_ankle_pose.translation()}")

# For each timestep, use FK to get consistent link poses
for t in range(balance_steps + 1):
    # Create temporary values for this timestep
    fk_values_t = gtsam.Values()
    
    # Set all joint angles to zero (standing pose)
    for j in range(num_joints):
        fk_values_t.insert(gtd.JointAngleKey(j, t), 0.0)
    
    # Set left ankle COM at desired position
    fk_values_t.insert(gtd.PoseKey(left_ankle_link.id(), t), desired_ankle_pose)
    
    # Run FK with left ankle as prior
    fk_result = robot.forwardKinematics(fk_values_t, t, left_ankle_link.name())
    
    # Copy FK results to init_values for all links
    for link in robot.links():
        pose_key = gtd.PoseKey(link.id(), t)
        if init_values.exists(pose_key):
            init_values.erase(pose_key)
        init_values.insert(pose_key, gtd.Pose(fk_result, link.id(), t))

# Update all twist values to zero (standing still)
zero_twist_vectorx = np.zeros(6)

for t in range(balance_steps + 1):
    for link in robot.links():
        twist_key = gtd.TwistKey(link.id(), t)
        if init_values.exists(twist_key):
            init_values.erase(twist_key)
        init_values.insert(twist_key, zero_twist_vectorx)

print(f"\nInitialized {init_values.size()} variables with FK-consistent poses")
print(f"Pelvis height at t=0: {gtd.Pose(init_values, pelvis_link.id(), 0).translation()[2]:.3f}m")
print(f"Left ankle height at t=0: {gtd.Pose(init_values, left_ankle_link.id(), 0).translation()[2]:.3f}m")
print(f"Right ankle height at t=0: {gtd.Pose(init_values, right_ankle_link.id(), 0).translation()[2]:.3f}m")

Initializing kinematically consistent poses using FK...
Using left_ankle_roll_link as prior at position: [0.   0.1  0.03]

Initialized 63812 variables with FK-consistent poses
Pelvis height at t=0: 0.727m
Left ankle height at t=0: 0.030m
Right ankle height at t=0: 0.030m

Initialized 63812 variables with FK-consistent poses
Pelvis height at t=0: 0.727m
Left ankle height at t=0: 0.030m
Right ankle height at t=0: 0.030m


## 7.1 Analyze Initial Error Before Optimization

Before running the optimization, let's analyze the factor graph errors on the initial zero values to understand the starting point of the problem.

In [11]:
# Analyze factor contributions to INITIAL error (before optimization)
print("Analyzing factor contributions to INITIAL error (before optimization)...")
print(f"Initial total error: {graph.error(init_values):.6e}\n")

# Get all factors and their errors on initial values
initial_factor_errors = []
initial_factor_types = []

for i in range(graph.size()):
    factor = graph.at(i)
    error = factor.error(init_values)
    initial_factor_errors.append(error)
    
    # Get the actual factor type name using Python's type system
    factor_type = type(factor).__name__
    initial_factor_types.append(factor_type)

# Convert to numpy arrays
initial_factor_errors = np.array(initial_factor_errors)
initial_factor_types = np.array(initial_factor_types)

# Group by factor type
unique_types = np.unique(initial_factor_types)
type_errors = {}
type_counts = {}

for ftype in unique_types:
    mask = initial_factor_types == ftype
    type_errors[ftype] = np.sum(initial_factor_errors[mask])
    type_counts[ftype] = np.sum(mask)

# Sort by error contribution
sorted_types = sorted(type_errors.keys(), key=lambda x: type_errors[x], reverse=True)

print(f"Factor Error Analysis (Initial Values):")
print(f"Total factors: {len(initial_factor_errors)}")
print(f"Total error: {np.sum(initial_factor_errors):.6e}")
print(f"\nError by factor type:")
for ftype in sorted_types:
    print(f"  {ftype:40s}: {type_errors[ftype]:12.6e} ({type_counts[ftype]:4d} factors, avg: {type_errors[ftype]/type_counts[ftype]:.6e})")

# Create visualization with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Error by Factor Type (Initial)',
        'Number of Factors by Type',
        'Average Error per Factor by Type (Initial)',
        'Error Distribution (Log Scale)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "histogram"}]]
)

# Plot 1: Total error by factor type
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf']
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Total Error'),
    row=1, col=1
)

# Plot 2: Number of factors by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Count'),
    row=1, col=2
)

# Plot 3: Average error per factor by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t]/type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Avg Error'),
    row=2, col=1
)

# Plot 4: Error distribution histogram (log scale)
fig.add_trace(
    go.Histogram(x=np.log10(initial_factor_errors[initial_factor_errors > 0]),
                 nbinsx=50,
                 marker_color='#1f77b4',
                 name='Error Distribution'),
    row=2, col=2
)

# Update layout
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(title_text="Log10(Error)", row=2, col=2)

fig.update_yaxes(type="log", title_text="Total Error", row=1, col=1)
fig.update_yaxes(title_text="Number of Factors", row=1, col=2)
fig.update_yaxes(type="log", title_text="Average Error", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="Factor Error Analysis (Initial Values - Before Optimization)",
    showlegend=False
)

fig.show()

Analyzing factor contributions to INITIAL error (before optimization)...
Initial total error: 1.993457e+12

Factor Error Analysis (Initial Values):
Total factors: 65622
Total error: 1.993457e+12

Error by factor type:
  NonlinearFactor                         : 1.993457e+12 (58047 factors, avg: 3.434212e+07)
  ContactHeightFactor                     : 1.159410e-25 ( 602 factors, avg: 1.925930e-28)
  PriorFactorDouble                       : 0.000000e+00 (6969 factors, avg: 0.000000e+00)
  PriorFactorPose3                        : 0.000000e+00 (   2 factors, avg: 0.000000e+00)
  PriorFactorVector                       : 0.000000e+00 (   2 factors, avg: 0.000000e+00)
Factor Error Analysis (Initial Values):
Total factors: 65622
Total error: 1.993457e+12

Error by factor type:
  NonlinearFactor                         : 1.993457e+12 (58047 factors, avg: 3.434212e+07)
  ContactHeightFactor                     : 1.159410e-25 ( 602 factors, avg: 1.925930e-28)
  PriorFactorDouble              

In [12]:
# Print top factors with largest INITIAL errors
number_of_factors = 30
print("\n" + "="*80)
print(f"TOP {number_of_factors} FACTORS WITH LARGEST INITIAL ERRORS")
print("="*80)

# Create list of (error, factor_type, factor_index) tuples
initial_factor_info = [(initial_factor_errors[i], initial_factor_types[i], i) for i in range(len(initial_factor_errors))]

# Sort by error in descending order
initial_factor_info_sorted = sorted(initial_factor_info, key=lambda x: x[0], reverse=True)

# Print top factors
print(f"\n{'Rank':<6} {'Error':<15} {'Factor Type':<40} {'Factor Index':<12}")
print("-" * 80)
for rank, (error, ftype, idx) in enumerate(initial_factor_info_sorted[:number_of_factors], 1):
    # Get the keys associated with this factor
    factor = graph.at(idx)
    keys = factor.keys()
    key_str = ", ".join([gtd.GTDKeyFormatter(k) for k in keys])
    
    print(f"{rank:<6} {error:<15.6e} {ftype:<40} {idx:<12}")
    print(f"       Keys: {key_str}")
    print()

print("="*80)


TOP 30 FACTORS WITH LARGEST INITIAL ERRORS

Rank   Error           Factor Type                              Factor Index
--------------------------------------------------------------------------------
1      4.662848e+09    NonlinearFactor                          102         
       Keys: A[13]0, F[13](12)0, F[13](13)0, F[13](18)0, V[13]0, p[13]0

2      4.662848e+09    NonlinearFactor                          297         
       Keys: A[13]1, F[13](12)1, F[13](13)1, F[13](18)1, V[13]1, p[13]1

3      4.662848e+09    NonlinearFactor                          492         
       Keys: A[13]2, F[13](12)2, F[13](13)2, F[13](18)2, V[13]2, p[13]2

4      4.662848e+09    NonlinearFactor                          687         
       Keys: A[13]3, F[13](12)3, F[13](13)3, F[13](18)3, V[13]3, p[13]3

5      4.662848e+09    NonlinearFactor                          882         
       Keys: A[13]4, F[13](12)4, F[13](13)4, F[13](18)4, V[13]4, p[13]4

6      4.662848e+09    NonlinearFactor         

In [13]:
# Check contact initialization
print("\nContact Point Initialization Check:")
print("="*80)
for i, cp in enumerate(contact_points):
    link = cp.link
    offset = cp.point
    print(f"\nContact {i}: {link.name()}")
    print(f"  Link COM pose at t=0: {gtd.Pose(init_values, link.id(), 0).translation()}")
    print(f"  Contact offset in link frame: {offset}")
    
    # Compute contact point in world frame
    link_pose = gtd.Pose(init_values, link.id(), 0)
    contact_world = link_pose.transformFrom(offset)
    print(f"  Contact point in world: {contact_world}")
    print(f"  Expected z=0, actual z={contact_world[2]:.6f}")
    
    # Check if wrench is initialized
    wrench_key = gtd.ContactWrenchKey(link.id(), 0, 0)
    if init_values.exists(wrench_key):
        wrench = init_values.atVector(wrench_key)
        print(f"  Contact wrench: {wrench}")
    else:
        print(f"  WARNING: No contact wrench initialized!")


Contact Point Initialization Check:

Contact 0: left_ankle_roll_link
  Link COM pose at t=0: [0.   0.1  0.03]
  Contact offset in link frame: [ 0.    0.   -0.03]
  Contact point in world: [0.  0.1 0. ]
  Expected z=0, actual z=0.000000
  Contact wrench: [0. 0. 0. 0. 0. 0.]

Contact 1: right_ankle_roll_link
  Link COM pose at t=0: [ 0.         -0.13701291  0.03      ]
  Contact offset in link frame: [ 0.    0.   -0.03]
  Contact point in world: [ 0.00000000e+00 -1.37012910e-01  2.77555756e-17]
  Expected z=0, actual z=0.000000
  Contact wrench: [0. 0. 0. 0. 0. 0.]


## 7.2 Run Optimization

In [14]:
print("\nStarting optimization...")

# Optimize using Levenberg-Marquardt
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")
params.setAbsoluteErrorTol(1e-6)
params.setRelativeErrorTol(1e-6)
params.setMaxIterations(300)

optimizer = gtsam.LevenbergMarquardtOptimizer(graph, init_values, params)
result_bal = optimizer.optimize()

print(f"\nOptimization complete!")
print(f"Final error: {graph.error(result_bal):.6e}")
print(f"Result contains {result_bal.size()} values")


Starting optimization...
Initial error: 
Optimization complete!
Final error: 1.916207e+11
Result contains 63812 values
1.99346e+12, values: 63812
iter      cost      cost_change    lambda  success iter_time
   0          inf            0      1e-05      0       0.87
iter      cost      cost_change    lambda  success iter_time
   0          inf            0     0.0001      0       0.83
iter      cost      cost_change    lambda  success iter_time
   0          inf            0      0.001      0       0.76
iter      cost      cost_change    lambda  success iter_time
   0      2.3e+14     -2.3e+14       0.01      1        1.1
iter      cost      cost_change    lambda  success iter_time
   0      3.8e+14     -3.8e+14        0.1      1        1.1
iter      cost      cost_change    lambda  success iter_time
   0        5e+12       -3e+12          1      1        1.1
iter      cost      cost_change    lambda  success iter_time
   0      5.5e+14     -5.5e+14         10      1        1.1
iter  

In [15]:
# Analyze factor contributions to FINAL error (after optimization)
print("Analyzing factor contributions to FINAL error (after optimization)...")
print(f"Final total error: {graph.error(result_bal):.6e}\n")

# Get all factors and their errors on optimized values
factor_errors = []
factor_types = []

for i in range(graph.size()):
    factor = graph.at(i)
    error = factor.error(result_bal)
    factor_errors.append(error)
    
    # Get the actual factor type name using Python's type system
    factor_type = type(factor).__name__
    factor_types.append(factor_type)

# Convert to numpy arrays
factor_errors = np.array(factor_errors)
factor_types = np.array(factor_types)

# Group by factor type
unique_types = np.unique(factor_types)
type_errors = {}
type_counts = {}

for ftype in unique_types:
    mask = factor_types == ftype
    type_errors[ftype] = np.sum(factor_errors[mask])
    type_counts[ftype] = np.sum(mask)

# Sort by error contribution
sorted_types = sorted(type_errors.keys(), key=lambda x: type_errors[x], reverse=True)

print(f"Factor Error Analysis (Optimized Values):")
print(f"Total factors: {len(factor_errors)}")
print(f"Total error: {np.sum(factor_errors):.6e}")
print(f"\nError by factor type:")
for ftype in sorted_types:
    print(f"  {ftype:40s}: {type_errors[ftype]:12.6e} ({type_counts[ftype]:4d} factors, avg: {type_errors[ftype]/type_counts[ftype]:.6e})")

# Create visualization with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Total Error by Factor Type (Optimized)',
        'Number of Factors by Type',
        'Average Error per Factor by Type (Optimized)',
        'Error Distribution (Log Scale)'
    ),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "histogram"}]]
)

# Plot 1: Total error by factor type
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#bcbd22', '#17becf']
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Total Error'),
    row=1, col=1
)

# Plot 2: Number of factors by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Count'),
    row=1, col=2
)

# Plot 3: Average error per factor by type
fig.add_trace(
    go.Bar(x=sorted_types, y=[type_errors[t]/type_counts[t] for t in sorted_types],
           marker_color=colors[:len(sorted_types)],
           name='Avg Error'),
    row=2, col=1
)

# Plot 4: Error distribution histogram (log scale)
fig.add_trace(
    go.Histogram(x=np.log10(factor_errors[factor_errors > 0]),
                 nbinsx=50,
                 marker_color='#1f77b4',
                 name='Error Distribution'),
    row=2, col=2
)

# Update layout
fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=1, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(title_text="Log10(Error)", row=2, col=2)

fig.update_yaxes(type="log", title_text="Total Error", row=1, col=1)
fig.update_yaxes(title_text="Number of Factors", row=1, col=2)
fig.update_yaxes(type="log", title_text="Average Error", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.update_layout(
    height=800,
    title_text="Factor Error Analysis (Optimized Values - After Optimization)",
    showlegend=False
)

fig.show()

Analyzing factor contributions to FINAL error (after optimization)...
Final total error: 1.916207e+11

Factor Error Analysis (Optimized Values):
Total factors: 65622
Total error: 1.916207e+11

Error by factor type:
  NonlinearFactor                         : 1.916203e+11 (58047 factors, avg: 3.301123e+06)
  PriorFactorDouble                       : 3.819788e+05 (6969 factors, avg: 5.481113e+01)
  PriorFactorPose3                        : 1.410568e+04 (   2 factors, avg: 7.052841e+03)
  ContactHeightFactor                     : 1.268434e+04 ( 602 factors, avg: 2.107034e+01)
  PriorFactorVector                       : 1.353800e+03 (   2 factors, avg: 6.769001e+02)


## 7.3 Analyze Final Error After Optimization

In [16]:
# Print top factors with largest FINAL errors (after optimization)
number_of_factors = 30
print("\n" + "="*80)
print(f"TOP {number_of_factors} FACTORS WITH LARGEST FINAL ERRORS (AFTER OPTIMIZATION)")
print("="*80)

# Create list of (error, factor_type, factor_index) tuples
factor_info = [(factor_errors[i], factor_types[i], i) for i in range(len(factor_errors))]

# Sort by error in descending order
factor_info_sorted = sorted(factor_info, key=lambda x: x[0], reverse=True)

# Print top 15
print(f"\n{'Rank':<6} {'Error':<15} {'Factor Type':<40} {'Factor Index':<12}")
print("-" * 80)
for rank, (error, ftype, idx) in enumerate(factor_info_sorted[:number_of_factors], 1):
    # Get the keys associated with this factor
    factor = graph.at(idx)
    keys = factor.keys()
    key_str = ", ".join([gtd.GTDKeyFormatter(k) for k in keys])
    
    print(f"{rank:<6} {error:<15.6e} {ftype:<40} {idx:<12}")
    print(f"       Keys: {key_str}")
    print()

print("="*80)


TOP 30 FACTORS WITH LARGEST FINAL ERRORS (AFTER OPTIMIZATION)

Rank   Error           Factor Type                              Factor Index
--------------------------------------------------------------------------------
1      5.494563e+10    NonlinearFactor                          90          
       Keys: p[12]0, C[12](0)0

2      2.815901e+10    NonlinearFactor                          285         
       Keys: p[12]1, C[12](0)1

3      2.063995e+10    NonlinearFactor                          480         
       Keys: p[12]2, C[12](0)2

4      1.592729e+10    NonlinearFactor                          675         
       Keys: p[12]3, C[12](0)3

5      1.245486e+10    NonlinearFactor                          870         
       Keys: p[12]4, C[12](0)4

6      9.638972e+09    NonlinearFactor                          1065        
       Keys: p[12]5, C[12](0)5

7      7.270348e+09    NonlinearFactor                          1260        
       Keys: p[12]6, C[12](0)6

8      6.972167

In [ ]:
gtd.GTDKeyFormatter(27583465385885697)

## 7.4 Visualize Optimized Torques

Visualize the optimized joint torques across all timesteps to verify the control effort.

In [ ]:
# Extract torques for all joints across the trajectory
time_steps = np.arange(balance_steps + 1) * balance_dt
torques = np.zeros((num_joints, balance_steps + 1))

# Extract torque values from the optimized result
for t in range(balance_steps + 1):
    for j in range(num_joints):
        torque_key = gtd.TorqueKey(j, t)
        if result_bal.exists(torque_key):
            torques[j, t] = result_bal.atDouble(torque_key)

# Get joint names for plotting
joint_names = [joint.name() for joint in robot.joints()]

# Create subplots for each joint
num_rows = (num_joints + 2) // 3  # 3 columns
num_cols = min(3, num_joints)

fig = make_subplots(
    rows=num_rows, cols=num_cols,
    subplot_titles=joint_names,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    shared_xaxes=True
)

# Plot torques for each joint
for j in range(num_joints):
    row = (j // 3) + 1
    col = (j % 3) + 1
    
    fig.add_trace(
        go.Scatter(
            x=time_steps,
            y=torques[j],
            mode='lines',
            name=joint_names[j],
            line=dict(width=2),
            showlegend=False
        ),
        row=row, col=col
    )
    
    # Add zero reference line
    fig.add_hline(
        y=0, 
        line_dash="dash", 
        line_color="gray", 
        opacity=0.5, 
        row=row, col=col
    )
    
    # Update y-axis label
    fig.update_yaxes(title_text="Torque (Nm)", row=row, col=col)

# Update x-axis labels (only for bottom row)
for col in range(1, num_cols + 1):
    fig.update_xaxes(title_text="Time (s)", row=num_rows, col=col)

fig.update_layout(
    height=300 * num_rows,
    title_text="Optimized Joint Torques Over Time",
    showlegend=False
)

fig.show()

# Print torque statistics
print("\n=== Joint Torque Statistics ===")
print(f"{'Joint Name':<30} {'Mean (Nm)':<12} {'Max (Nm)':<12} {'Min (Nm)':<12} {'Std (Nm)':<12} {'RMS (Nm)':<12}")
print("-" * 100)

for j in range(num_joints):
    mean_torque = np.mean(torques[j])
    max_torque = np.max(np.abs(torques[j]))
    min_torque = np.min(torques[j])
    std_torque = np.std(torques[j])
    rms_torque = np.sqrt(np.mean(torques[j]**2))
    
    print(f"{joint_names[j]:<30} {mean_torque:>11.4f} {max_torque:>11.4f} {min_torque:>11.4f} {std_torque:>11.4f} {rms_torque:>11.4f}")

print("\n=== Overall Statistics ===")
print(f"Total RMS torque across all joints: {np.sqrt(np.mean(torques**2)):.4f} Nm")
print(f"Maximum absolute torque: {np.max(np.abs(torques)):.4f} Nm")
print(f"Average torque magnitude: {np.mean(np.abs(torques)):.4f} Nm")

## 7.5 Visualize Contact Point Heights

Check that contact points remain on the ground throughout the trajectory.

In [17]:
# Extract and visualize contact point heights over time
time_steps = np.arange(balance_steps + 1) * balance_dt

# Create figure for contact point heights
fig = make_subplots(
    rows=len(contact_points), cols=1,
    subplot_titles=[f"Contact Point: {cp.link.name()}" for cp in contact_points],
    vertical_spacing=0.1,
    shared_xaxes=True
)

contact_heights = {i: [] for i in range(len(contact_points))}

# Calculate contact point heights for each timestep
for t in range(balance_steps + 1):
    for i, point_on_link in enumerate(contact_points):
        link_id = point_on_link.link.id()
        # Get link pose at timestep t
        link_pose = gtd.Pose(result_bal, link_id, t)
        # Transform contact point to world frame
        contact_point_world = link_pose.transformFrom(point_on_link.point)
        # Get z-coordinate (height)
        contact_heights[i].append(contact_point_world[2])

# Plot heights for each contact point
for i, point_on_link in enumerate(contact_points):
    fig.add_trace(
        go.Scatter(
            x=time_steps,
            y=contact_heights[i],
            mode='lines',
            name=point_on_link.link.name(),
            line=dict(width=2, color='blue'),
            showlegend=False
        ),
        row=i+1, col=1
    )
    
    # Add ground plane reference
    fig.add_hline(
        y=ground_plane_height, 
        line_dash="dash", 
        line_color="green", 
        opacity=0.7, 
        row=i+1, col=1,
        annotation_text="Ground" if i == 0 else None
    )
    
    # Update y-axis
    fig.update_yaxes(title_text="Height (m)", row=i+1, col=1)

# Update layout
fig.update_xaxes(title_text="Time (s)", row=len(contact_points), col=1)
fig.update_layout(
    height=300*len(contact_points),
    title_text="Contact Point Heights Over Time",
    showlegend=False
)

fig.show()

# Print statistics
print("\n=== Contact Point Height Statistics ===")
for i, point_on_link in enumerate(contact_points):
    heights = np.array(contact_heights[i])
    mean_height = np.mean(heights)
    max_height = np.max(heights)
    min_height = np.min(heights)
    std_height = np.std(heights)
    print(f"{point_on_link.link.name():30s}: Mean={mean_height:.6f}m, "
          f"Min={min_height:.6f}m, Max={max_height:.6f}m, Std={std_height:.6f}m")


=== Contact Point Height Statistics ===
left_ankle_roll_link          : Mean=0.006489m, Min=-0.003976m, Max=0.007844m, Std=0.000815m
right_ankle_roll_link         : Mean=0.006384m, Min=0.002767m, Max=0.014227m, Std=0.000876m


## 8. Linearize and Create LQR Policy (Bayes Net)

In [ ]:
print("Linearizing factor graph around optimized trajectory...")

# Linearize the nonlinear factor graph
gaussian_graph_bal = graph.linearize(result_bal)

print(f"Linearized graph contains {gaussian_graph_bal.size()} Gaussian factors")

# Create elimination ordering (backward in time)
# Controls (torques) first, then states (angles, velocities, accelerations)
print("Creating backward elimination ordering...")
ordering_bal = gtsam.Ordering()
contact_link_ids = [cp.link.id() for cp in contact_points]

num_links = robot.numLinks()

# Add variables from t = balance_steps down to t = 0
for t in reversed(range(balance_steps + 1)):
    # Torques (controls) - eliminate first
    for j in range(num_joints):
        key = gtd.TorqueKey(j, t)
        if result_bal.exists(key):
            ordering_bal.push_back(key)
    
    # Contact wrenches (state) - eliminate after controls
    # ContactWrenchKey signature is (link_id, 0, timestep)
    for link_id in contact_link_ids:
        key = gtd.ContactWrenchKey(link_id, 0, t)
        if result_bal.exists(key):
            ordering_bal.push_back(key)
    
    # Joint angles (state)
    for j in range(num_joints):
        key = gtd.JointAngleKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.PoseKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    
    # Accelerations (state)
    for j in range(num_joints):
        key = gtd.JointAccelKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistAccelKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)

    # Wrenches (state)
    for i in range(num_links):
        for j in range(num_joints):
            key = gtd.WrenchKey(i, j, t)
            if result_bal.exists(key): 
                ordering_bal.push_back(key)

    # Velocities (state)
    for j in range(num_joints):
        key = gtd.JointVelKey(j, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)
    for i in range(num_links):
        key = gtd.TwistKey(i, t)
        if result_bal.exists(key): 
            ordering_bal.push_back(key)

print(f"Elimination ordering created with {ordering_bal.size()} keys")

# Verify ordering
if ordering_bal.size() != result_bal.size():
    print(f"WARNING: Ordering size ({ordering_bal.size()}) != result size ({result_bal.size()})")
else:
    print("Ordering size matches result size ✓")

In [ ]:
# Eliminate to get the Bayes Net (LQR policy)
print("\nPerforming sequential elimination...")
policy_bayes_net_bal = gaussian_graph_bal.eliminateSequential(ordering_bal)

print(f"Bayes Net created with {policy_bayes_net_bal.size()} conditionals")
print("LQR policy ready!")

In [ ]:
gtd.GTDKeyFormatter(24233231981216040)

## 9. MuJoCo Simulation Setup

In [ ]:
# Initialize MuJoCo model and data
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

print(f"MuJoCo model loaded")
print(f"Number of MuJoCo qpos: {model.nq}")
print(f"Number of MuJoCo qvel: {model.nv}")
print(f"Number of MuJoCo actuators: {model.nu}")

# Create mapping from GTDynamics joint order to MuJoCo joint indices
# MuJoCo uses different ordering than URDF
gtd_to_mujoco_pos = {}  # For qpos addresses
gtd_to_mujoco_vel = {}  # For qvel addresses
mujoco_joint_names = []

for i in range(model.njnt):
    joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    if joint_name:
        mujoco_joint_names.append(joint_name)

print(f"\nMuJoCo joint names ({len(mujoco_joint_names)}):")
for i, name in enumerate(mujoco_joint_names):
    print(f"  {i}: {name}")

# Map GTDynamics joints to MuJoCo joint addresses
for gtd_idx, joint in enumerate(robot.joints()):
    gtd_joint_name = joint.name()
    # Find corresponding MuJoCo joint
    for mj_idx, mj_name in enumerate(mujoco_joint_names):
        if gtd_joint_name == mj_name:
            # Get the qpos and qvel addresses for this joint
            joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, mj_name)
            qpos_addr = model.jnt_qposadr[joint_id]
            qvel_addr = model.jnt_dofadr[joint_id]  # Use dofadr for velocity
            gtd_to_mujoco_pos[gtd_idx] = qpos_addr
            gtd_to_mujoco_vel[gtd_idx] = qvel_addr
            break

print(f"\nMapped {len(gtd_to_mujoco_pos)} joints from GTDynamics to MuJoCo")
print(f"Position mapping size: {len(gtd_to_mujoco_pos)}")
print(f"Velocity mapping size: {len(gtd_to_mujoco_vel)}")

# Create mapping from GTDynamics link IDs to MuJoCo body IDs
gtd_to_mujoco_body = {}
for link in robot.links():
    link_name = link.name()
    gtd_link_id = link.id()
    # Find corresponding MuJoCo body
    try:
        mj_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, link_name)
        gtd_to_mujoco_body[gtd_link_id] = mj_body_id
    except:
        print(f"Warning: Could not find MuJoCo body for link '{link_name}'")

print(f"\nMapped {len(gtd_to_mujoco_body)} links from GTDynamics to MuJoCo bodies")


## 10. Run TVLQR Feedback Control Simulation

In [ ]:
# Reset MuJoCo simulation to clean state
mujoco.mj_resetData(model, data)

# Set initial conditions - base position at (0, 0, 0.8) and zero joint angles
#data.qpos[:3] = np.array([0.0, 0.0, 0.])  # Base position
#data.qpos[3:7] = np.array([1.0, 0.0, 0.0, 0.0])  # Base quaternion (identity: w=1, x=0, y=0, z=0)

# Set joint angles
for gtd_idx in range(num_joints):
    if gtd_idx in gtd_to_mujoco_pos:
        mj_qpos_addr = gtd_to_mujoco_pos[gtd_idx]
        data.qpos[mj_qpos_addr] = initial_joint_angles[gtd_idx]

# Define the two different timesteps
control_dt = balance_dt   # 0.05s (20 Hz) - derived from your factor graph
physics_dt = 0.01        # 0.002s (500 Hz) - standard for stable contact physics

# Tell MuJoCo to use the high-frequency physics step
model.opt.timestep = physics_dt

# Calculate how many physics steps to take per control step
sim_substeps = int(control_dt / physics_dt)

print("Starting MuJoCo simulation with TVLQR feedback control...")

# Rendering parameters
VIDEO_WIDTH = 800
VIDEO_HEIGHT = 600
render_freq = 30  # fps
frames = []

# Data storage for tracking deviations and control
tracking_data = {
    'time': [],
    'joint_angle_deviations': [[] for _ in range(num_joints)],
    'joint_vel_deviations': [[] for _ in range(num_joints)],
    'torque_corrections': [[] for _ in range(num_joints)],
    'torque_references': [[] for _ in range(num_joints)],
    'torque_totals': [[] for _ in range(num_joints)],
    'base_position': [],
    'base_orientation': [],
    # Add planned and actual values
    'planned_joint_angles': [[] for _ in range(num_joints)],
    'actual_joint_angles': [[] for _ in range(num_joints)],
    'planned_base_position': [],
    'actual_base_position': []
}

with mujoco.Renderer(model, width=VIDEO_WIDTH, height=VIDEO_HEIGHT) as renderer:
    
    for t in range(balance_steps):
        current_time = t * balance_dt
        
        # Get actual state from MuJoCo
        q_actual = np.zeros(num_joints)
        v_actual = np.zeros(num_joints)
        
        for gtd_idx in range(num_joints):
            if gtd_idx in gtd_to_mujoco_pos:
                mj_qpos_addr = gtd_to_mujoco_pos[gtd_idx]
                mj_qvel_addr = gtd_to_mujoco_vel[gtd_idx]
                q_actual[gtd_idx] = data.qpos[mj_qpos_addr]
                v_actual[gtd_idx] = data.qvel[mj_qvel_addr]
        
        # Get reference state from optimized trajectory
        q_ref = np.array([gtd.JointAngle(result_bal, j, t) for j in range(num_joints)])
        v_ref = np.array([gtd.JointVel(result_bal, j, t) for j in range(num_joints)])
        u_ref = np.array([gtd.Torque(result_bal, j, t) for j in range(num_joints)])
        
        # Get planned base position and twist
        planned_base_pose = gtd.Pose(result_bal, pelvis_link_id, t)
        planned_base_pos = planned_base_pose.translation()
        # Use GTDynamics helper function to get twist
        planned_base_twist = gtd.Twist(result_bal, pelvis_link_id, t)
        
        # Get actual base pose and twist from MuJoCo
        actual_base_pos = data.qpos[:3]  # x, y, z
        actual_base_quat = data.qpos[3:7]  # quaternion (w, x, y, z in MuJoCo format)
        actual_base_rot = gtsam.Rot3.Quaternion(actual_base_quat[0], actual_base_quat[1], 
                                                 actual_base_quat[2], actual_base_quat[3])
        actual_base_pose = gtsam.Pose3(actual_base_rot, actual_base_pos)
        
        # Get actual base twist (6D: linear + angular velocity)
        actual_base_twist = data.qvel[:6]  # First 6 elements are base velocities
        
        # Compute state deviations
        dx = gtsam.VectorValues()
        
        # Add joint state deviations
        for j in range(num_joints):
            dx.insert(gtd.JointAngleKey(j, t), np.array([q_actual[j] - q_ref[j]]))
            dx.insert(gtd.JointVelKey(j, t), np.array([v_actual[j] - v_ref[j]]))
        
        # Add pose and twist deviations for ALL links from MuJoCo
        for link_id in range(num_links):
            # Get planned pose and twist from optimization
            if result_bal.exists(gtd.PoseKey(link_id, t)):
                planned_link_pose = gtd.Pose(result_bal, link_id, t)
                
                # Get actual link pose from MuJoCo
                if link_id in gtd_to_mujoco_body:
                    mj_body_id = gtd_to_mujoco_body[link_id]
                    # MuJoCo provides body position and orientation
                    actual_link_pos = data.xpos[mj_body_id]  # 3D position
                    actual_link_quat = data.xquat[mj_body_id]  # quaternion (w, x, y, z)
                    actual_link_rot = gtsam.Rot3.Quaternion(actual_link_quat[0], actual_link_quat[1],
                                                             actual_link_quat[2], actual_link_quat[3])
                    actual_link_pose = gtsam.Pose3(actual_link_rot, actual_link_pos)
                else:
                    # Skip if body not found in MuJoCo
                    continue
                
                # Compute pose error in SE(3) tangent space
                pose_error = actual_link_pose.between(planned_link_pose)
                pose_error_vector = gtsam.Pose3.Logmap(pose_error)
                dx.insert(gtd.PoseKey(link_id, t), pose_error_vector)
            
            # Get planned twist and compute deviation
            if result_bal.exists(gtd.TwistKey(link_id, t)):
                planned_link_twist = gtd.Twist(result_bal, link_id, t)
                
                # Get actual link twist from MuJoCo
                if link_id in gtd_to_mujoco_body:
                    mj_body_id = gtd_to_mujoco_body[link_id]
                    # MuJoCo provides body linear and angular velocities
                    # Note: MuJoCo stores velocities in world frame
                    linear_vel = data.cvel[mj_body_id][:3]  # Linear velocity (world frame)
                    angular_vel = data.cvel[mj_body_id][3:]  # Angular velocity (world frame)
                    actual_link_twist = np.concatenate([angular_vel, linear_vel])  # GTDynamics uses [angular, linear]
                else:
                    # Skip if body not found in MuJoCo
                    continue
                
                twist_error = actual_link_twist - planned_link_twist
                dx.insert(gtd.TwistKey(link_id, t), twist_error)
        
        # Filter Bayes net to remove provided state variables
        provided_keys = set()
        for j in range(num_joints):
            provided_keys.add(gtd.JointAngleKey(j, t))
            provided_keys.add(gtd.JointVelKey(j, t))
        
        # Add ALL link pose and twist keys that we're providing from MuJoCo
        for link_id in range(num_links):
            if result_bal.exists(gtd.PoseKey(link_id, t)):
                provided_keys.add(gtd.PoseKey(link_id, t))
            if result_bal.exists(gtd.TwistKey(link_id, t)):
                provided_keys.add(gtd.TwistKey(link_id, t))
        
        filtered_bayes_net = gtsam.GaussianBayesNet()
        for i in range(policy_bayes_net_bal.size()):
            conditional = policy_bayes_net_bal.at(i)
            frontal_key = conditional.firstFrontalKey()
            if frontal_key not in provided_keys:
                filtered_bayes_net.push_back(conditional)
        
        # Compute optimal corrections using LQR policy
        optimal_corrections = filtered_bayes_net.optimize(dx)
        
        # Extract torque corrections
        du = np.zeros(num_joints)
        for j in range(num_joints):
            du[j] = optimal_corrections.at(gtd.TorqueKey(j, t))[0]
        
        # Apply total torque
        u_total = u_ref + du
                
        # Set MuJoCo control
        for j in range(num_joints):
            if j < model.nu:  # Check if actuator exists
                data.ctrl[j] = u_total[j]
        
        # Store tracking data
        tracking_data['time'].append(current_time)
        # Base position/orientation (first 7 elements of qpos for floating base)
        base_pos = data.qpos[:3].copy()
        base_quat = data.qpos[3:7].copy()
        tracking_data['base_position'].append(base_pos)
        tracking_data['base_orientation'].append(base_quat)
        tracking_data['planned_base_position'].append(planned_base_pos)
        tracking_data['actual_base_position'].append(base_pos)
        
        for j in range(num_joints):
            tracking_data['joint_angle_deviations'][j].append(q_actual[j] - q_ref[j])
            tracking_data['joint_vel_deviations'][j].append(v_actual[j] - v_ref[j])
            tracking_data['torque_corrections'][j].append(du[j])
            tracking_data['torque_references'][j].append(u_ref[j])
            tracking_data['torque_totals'][j].append(u_total[j])
            tracking_data['planned_joint_angles'][j].append(q_ref[j])
            tracking_data['actual_joint_angles'][j].append(q_actual[j])
        
        # Simulate and render frames for this control timestep
        for _ in range(sim_substeps):
            mujoco.mj_step(model, data)

        # Render frame at the end of the control step
        renderer.update_scene(data)
        frames.append(renderer.render())
        
        if (t + 1) % 10 == 0:
            print(f"Progress: {t+1}/{balance_steps} steps")

print("Simulation complete!")

# Show the video
media.show_video(frames, fps=render_freq, width=VIDEO_WIDTH, height=VIDEO_HEIGHT)

## 11. Visualize Joint Deviations (All Joints)

In [ ]:
# Prepare data
time = tracking_data['time']
joint_names = [joint.name() for joint in robot.joints()]

# Create figure with subplots for joint angle deviations
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's angle deviation
for j in range(num_joints):
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['joint_angle_deviations'][j]),
            mode='lines',
            name=joint_names[j],
            line=dict(width=2),
            showlegend=False
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig.update_yaxes(title_text="Deg", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Joint Angle Deviations (Actual - Planned)",
    showlegend=False
)

fig.show()

# Create figure for joint velocity deviations
fig2 = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's velocity deviation
for j in range(num_joints):
    fig2.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['joint_vel_deviations'][j]),
            mode='lines',
            name=joint_names[j],
            line=dict(width=2, color='red'),
            showlegend=False
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig2.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig2.update_yaxes(title_text="Deg/s", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig2.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig2.update_layout(
    height=300*num_joints,
    title_text="Joint Velocity Deviations (Actual - Planned)",
    showlegend=False
)

fig2.show()

# Print statistics
print("\n=== Deviation Statistics ===")
for j in range(num_joints):
    max_angle_dev = max(np.abs(tracking_data['joint_angle_deviations'][j]))
    max_vel_dev = max(np.abs(tracking_data['joint_vel_deviations'][j]))
    print(f"{joint_names[j]:30s}: Angle={np.rad2deg(max_angle_dev):8.4f} deg, Vel={np.rad2deg(max_vel_dev):8.4f} deg/s")

## 12. Planned vs Actual Joint Angles

In [ ]:
# Create figure comparing planned vs actual joint angles
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint
for j in range(num_joints):
    # Planned angles
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['planned_joint_angles'][j]),
            mode='lines',
            name='Planned',
            line=dict(width=2, color='blue', dash='dash'),
            legendgroup='planned',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Actual angles
    fig.add_trace(
        go.Scatter(
            x=time,
            y=np.rad2deg(tracking_data['actual_joint_angles'][j]),
            mode='lines',
            name='Actual',
            line=dict(width=2, color='red'),
            legendgroup='actual',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Update y-axis
    fig.update_yaxes(title_text="Deg", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Planned vs Actual Joint Angles",
    showlegend=True,
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)

fig.show()

## 13. Applied Torques Visualization

In [ ]:
# Visualize applied torques for all joints
fig = make_subplots(
    rows=num_joints, cols=1,
    subplot_titles=[f"Joint {i}: {joint_names[i]}" for i in range(num_joints)],
    vertical_spacing=0.02,
    shared_xaxes=True
)

# Add traces for each joint's torque
for j in range(num_joints):
    # Total torque (reference + correction)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_totals'][j],
            mode='lines',
            name='Total',
            line=dict(width=2, color='black'),
            legendgroup='total',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Reference torque (from optimization)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_references'][j],
            mode='lines',
            name='Reference',
            line=dict(width=1.5, color='blue', dash='dash'),
            legendgroup='reference',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Correction torque (from LQR feedback)
    fig.add_trace(
        go.Scatter(
            x=time,
            y=tracking_data['torque_corrections'][j],
            mode='lines',
            name='Correction',
            line=dict(width=1.5, color='red', dash='dot'),
            legendgroup='correction',
            showlegend=(j==0)
        ),
        row=j+1, col=1
    )
    
    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=j+1, col=1)
    
    # Update y-axis
    fig.update_yaxes(title_text="N·m", row=j+1, col=1, title_font=dict(size=10))

# Update layout
fig.update_xaxes(title_text="Time (s)", row=num_joints, col=1)
fig.update_layout(
    height=300*num_joints,
    title_text="Applied Torques - All Joints (TVLQR Control)",
    showlegend=True,
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)

fig.show()

# Print torque statistics
print("\n=== Applied Torque Statistics ===")
for j in range(num_joints):
    max_total = max(np.abs(tracking_data['torque_totals'][j]))
    max_ref = max(np.abs(tracking_data['torque_references'][j]))
    max_corr = max(np.abs(tracking_data['torque_corrections'][j]))
    mean_total = np.mean(np.abs(tracking_data['torque_totals'][j]))
    print(f"{joint_names[j]:30s}: Max Total={max_total:8.4f} N·m, Max Ref={max_ref:8.4f} N·m, Max Corr={max_corr:8.4f} N·m, Mean |Total|={mean_total:8.4f} N·m")